# ExoBaby — Long-Tailed Distribution Validation (Qwen3.5-9B via CraftX API)

**Model:** `Qwen3.5 9B` via CraftX OpenAI-compatible API
**Inference style:** Closed-list detection (ExoBaby_LongTailed_Colab style)
**Vocabulary:** User-configurable CDI path (set `VOCAB_CSV` in Step 1)
**Hardware:** Runs on **Standard CPU** runtime (no GPU required — fast multi-threaded API requests)

## Workflow
1. Mount Drive and set API Key & paths
2. Collect all frames — **excluding** the 1,000 frames used to mine the CDI vocabulary
3. Load CDI vocabulary (377 or 551 items) sorted by frequency
4. Pre-flight diagnostic check + multi-threaded CraftX API closed-list detection
5. Fit power-law, generate figures, OOI coverage, provenance (identical to `ExoBaby_LongTailed_Colab`)

---

## Step 0 — Clone repo & install packages

In [ ]:
REPO_URL  = 'https://github.com/tanzim12911/ExoBaby-pipeline.git'
REPO_NAME = 'ExoBaby-pipeline'

import os, subprocess
if not os.path.isdir(f'/content/{REPO_NAME}'):
    subprocess.run(['git', 'clone', REPO_URL, f'/content/{REPO_NAME}'], check=True)
    print('Repo cloned.')
else:
    subprocess.run(['git', '-C', f'/content/{REPO_NAME}', 'pull'], check=True)
    print('Repo updated (git pull).')
os.chdir(f'/content/{REPO_NAME}')
print('Working directory:', os.getcwd())

In [ ]:
# Lightweight dependencies (no heavy PyTorch / CUDA downloads needed)
!pip install -q requests pillow pandas tqdm matplotlib powerlaw scipy
print('All packages ready.')

---
## Step 1 — Mount Drive & Configure API Key & Paths

Set your `CRAFTX_API_KEY` and configuration options below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── CONFIGURATION ─────────────────────────────────────────
CRAFTX_API_KEY = 'YOUR_CRAFTX_API_KEY_HERE'
MODEL_NAME     = 'Qwen3.5 9B'

FRAMES_DIR  = '/content/drive/MyDrive/data/filtered'
OUTPUT_BASE = '/content/drive/MyDrive/ExoBaby-results'
RUN_TAG     = 'qwen35_9b_craftx'   # output subfolder tag

# CDI vocabulary — pick one:
VOCAB_CSV = 'CDI/Bangla/stage4_concrete_objects_qwen3_5_9B_CDI_377.csv'
# VOCAB_CSV = 'CDI/Bangla/stage4_concrete_objects_qwen3_5_9B_CDI_551.csv'

# Mining frame exclusion log (frames used to mine the CDI)
MINING_LOG  = 'CDI/Bangla/Qwen3_5_9B_mining_sampled_frames_log.txt'

MAX_WORKERS = 8       # Parallel API worker threads
PRUNE_STALE = False   # True = remove CSV rows for frames no longer on disk
# ──────────────────────────────────────────────────

import os
import pandas as pd
import detection.config as cfg
from run_detection import resolve_shared_dirs, resolve_analysis_dirs

shared_dirs   = resolve_shared_dirs(OUTPUT_BASE)
analysis_dirs = resolve_analysis_dirs(OUTPUT_BASE, RUN_TAG)

dirs = {
    'data':       shared_dirs['data'],
    'frame_data': shared_dirs['frame_data'],
    'results':    analysis_dirs['results'],
    'figures':    analysis_dirs['figures'],
}

vocab_tag     = os.path.splitext(os.path.basename(VOCAB_CSV))[0]
detection_csv = os.path.join(shared_dirs['frame_data'], f'detections_craftx_{vocab_tag}.csv')

print(f'CraftX Model    : {MODEL_NAME}')
print(f'Frames dir      : {FRAMES_DIR}')
print(f'Vocab CSV       : {VOCAB_CSV}')
print(f'Detection CSV   : {detection_csv}')
print(f'Analysis output : {OUTPUT_BASE}/analysis/{RUN_TAG}/')

if CRAFTX_API_KEY == 'YOUR_CRAFTX_API_KEY_HERE':
    print('\n  ⚠️ WARNING: Please replace CRAFTX_API_KEY with your real API key!')
else:
    print('\n  API Key set!')

---
## Step 2 — Collect frames & exclude mining set

All frames are collected, then the **1,000 frames used for word mining** are excluded.

In [ ]:
from run_detection import collect_frames
from detection.craftx_detector import load_mining_exclusions

all_frame_paths = collect_frames(FRAMES_DIR)
print(f'Total frames found: {len(all_frame_paths):,}')

mining_exclusions = load_mining_exclusions(MINING_LOG)
print(f'Mining exclusion set: {len(mining_exclusions):,}')

def _rel(path, base):
    return os.path.relpath(path, base).replace('\\', '/')

frame_paths = [
    p for p in all_frame_paths
    if _rel(p, FRAMES_DIR) not in mining_exclusions
]

excluded = len(all_frame_paths) - len(frame_paths)
print(f'Excluded mining frames : {excluded:,}')
print(f'Frames for API inference: {len(frame_paths):,}')

---
## Step 3 — Load CDI vocabulary

Categories are sorted by **`presence_count` descending**.

In [ ]:
if not os.path.exists(VOCAB_CSV):
    raise FileNotFoundError(f'CDI file not found: {VOCAB_CSV}')

df_vocab = pd.read_csv(VOCAB_CSV)
df_vocab = df_vocab.sort_values('presence_count', ascending=False).reset_index(drop=True)
df_vocab['english_label_clean'] = df_vocab['english_label'].str.strip().str.lower()

included_categories = df_vocab['english_label_clean'].tolist()
lemma_to_semantic   = (
    df_vocab.drop_duplicates(subset=['english_label_clean'])
            .set_index('english_label_clean')['category']
            .str.strip().str.lower()
            .to_dict()
)

print(f'Vocabulary : {VOCAB_CSV}')
print(f'Categories : {len(included_categories)}')
print(f'Domains    : {sorted(df_vocab["category"].unique())}')
print(f'Top 10     : {included_categories[:10]}')

---
## Step 4 — Pre-flight Diagnostic Check & CraftX API Detection

> **Diagnostic Check:** Runs a test request on the 1st frame and prints raw API status & response JSON.
> **Resumable:** Already-processed frames are skipped automatically if re-run.

In [ ]:
from detection.craftx_detector import run_craftx_detection

run_craftx_detection(
    frame_paths=frame_paths,
    included_categories=included_categories,
    output_csv=detection_csv,
    api_key=CRAFTX_API_KEY,
    model_name=MODEL_NAME,
    max_workers=MAX_WORKERS,
    prune_stale=PRUNE_STALE,
)

print(f'\nDetection CSV saved to: {detection_csv}')

In [ ]:
df_raw = pd.read_csv(detection_csv)
df_det_all = df_raw[df_raw['class_name'].notna() & (df_raw['class_name'] != '')].copy()

print(f'Total CSV rows         : {len(df_raw):,}')
print(f'Sentinel rows (no det) : {len(df_raw) - len(df_det_all):,}')
print(f'Total detections       : {len(df_det_all):,}')
print(f'Unique frames hit      : {df_det_all["frame_path"].nunique():,} / {len(frame_paths):,}')
print(f'Unique categories      : {df_det_all["class_name"].nunique()} / {len(included_categories)}')
print('\nTop 20 detected categories:')
print(df_det_all['class_name'].value_counts().head(20).to_string())

---
## Step 5 — Long-tailed distribution analysis

In [ ]:
from detection.analysis import build_category_summary, fit_power_law, print_summary

intermediate_csv = os.path.join(dirs['results'], f'long_tailed_dist_{vocab_tag}.csv')

df_cat = build_category_summary(
    detection_csv=detection_csv,
    included_categories=included_categories,
    lemma_to_semantic=lemma_to_semantic,
    total_frames=len(frame_paths),
    output_csv=intermediate_csv,
)

fit = fit_power_law(df_cat)
print(f'Power-law exponent alpha : {fit["alpha"]:.3f}')
print(f'xmin                     : {fit["xmin"]:.1f}')
print(f'R (PL vs lognormal)      : {fit["r_vs_lognormal"]:.3f}  p = {fit["p_vs_lognormal"]:.4f}')

---
## Step 6 — Figures

In [ ]:
import matplotlib.pyplot as plt
import detection.visualizer as vis
from detection.visualizer import plot_rank_bar, plot_loglog, plot_per_domain

def plot_per_domain_clean(df_cat, figures_dir: str, dataset_label: str = 'ExoBaby') -> str:
    present   = [d for d in vis.CDI_SEMANTIC_ORDER if d in df_cat['cdi_semantic'].values]
    ncols     = 4
    nrows     = -(-len(present) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.5 * nrows))
    axes_flat = axes.flatten() if nrows > 1 else list(axes)
    for ax, domain in zip(axes_flat, present):
        sub   = df_cat[(df_cat['cdi_semantic'] == domain) & (df_cat['proportion'] > 0)].sort_values('proportion', ascending=False)
        color = vis.CDI_SEMANTIC_COLORS.get(domain, '#8B9A9E')
        if len(sub) == 0:
            ax.text(0.5, 0.5, 'No Detections', ha='center', va='center', color='gray', fontsize=11)
            ax.set_xticks([])
        else:
            sub = sub.head(15)
            ax.bar(range(len(sub)), sub['proportion'], color=color, width=0.85, linewidth=0)
            ax.set_xticks(range(len(sub)))
            ax.set_xticklabels(sub['category'].tolist(), rotation=45, ha='right', fontsize=9)
        ax.set_title(domain.replace('_', ' ').title(), fontsize=11, color=color, fontweight='bold')
        vis._style_ax(ax)
    for ax in axes_flat[len(present):]:
        ax.set_visible(False)
    fig.suptitle(f'Per-Domain Object Distribution ({dataset_label})', fontsize=14, y=1.01)
    fig.tight_layout()
    out_path = vis._save_figure(fig, figures_dir, f'fig2_per_domain_{vocab_tag}.png')
    return out_path

vis.plot_per_domain = plot_per_domain_clean
total_detected = len(df_det_all)
dataset_label  = f'ExoBaby — CraftX Qwen3.5 9B ({vocab_tag})'

bar_path = plot_rank_bar(df_cat, fit, len(frame_paths), total_detected, dirs['figures'], dataset_label=dataset_label)
log_path = plot_loglog(fit['nonzero_df'], fit['counts_arr'], fit['alpha'], dirs['figures'])
dom_path = plot_per_domain(df_cat, dirs['figures'], dataset_label=dataset_label)
print(f'Saved figures:\n  - {bar_path}\n  - {log_path}\n  - {dom_path}')

---
## Step 7 — Results Summary

In [ ]:
print_summary(df_cat, fit, len(frame_paths), total_detected)

---
## Step 8 — Frame-level OOI Coverage

In [ ]:
import json, os

frames_with_detections = df_det_all['frame_path'].nunique()
total_frames           = len(frame_paths)
proportion             = frames_with_detections / total_frames if total_frames else 0.0

ooi_stats = {
    'run_tag':                RUN_TAG,
    'model':                  'CraftX Qwen3.5 9B API',
    'vocabulary':             os.path.basename(VOCAB_CSV),
    'total_held_out_frames':  total_frames,
    'frames_with_detections': frames_with_detections,
    'proportion':             round(proportion, 6),
    'proportion_pct':         f'{proportion:.2%}',
}

ooi_json = os.path.join(dirs['results'], f'ooi_coverage_{vocab_tag}.json')
with open(ooi_json, 'w', encoding='utf-8') as f:
    json.dump(ooi_stats, f, indent=2)
print(f'Saved OOI coverage to: {ooi_json}')

---
## Step 9 — Provenance Analysis

In [ ]:
from detection.analysis import build_provenance
from detection.visualizer import plot_provenance

provenance_csv = os.path.join(dirs['results'], f'provenance_top{cfg.TOP_N_PER_DOMAIN}_per_domain_{vocab_tag}.csv')
df_prov = build_provenance(detection_csv=detection_csv, df_cat=df_cat, output_csv=provenance_csv, top_n=cfg.TOP_N_PER_DOMAIN)
prov_path = plot_provenance(df_prov, cfg.TOP_N_PER_DOMAIN, dirs['figures'])
print(f'Saved provenance to: {provenance_csv}')